# Auto Annotation with Label Studio ML Backend

**Pipeline:** YOLO26n (detection) → SAM2 (segmentation) → Label Studio

The ML backend runs as a Flask server on `http://localhost:9090`.
Label Studio calls it automatically when you open a task to pre-annotate it.

---

## 1. Architecture

```
Label Studio (port 8080)
       │  POST /predict  (sends image URL)
       ▼
ML Backend (port 9090)
       │
       ├── YOLO26n  → bounding boxes + class labels
       │
       └── SAM2     → polygon masks (using YOLO boxes as prompts)
       │
       └── returns PolygonLabels (or RectangleLabels fallback) to Label Studio
```

---

## 2. Dependencies

- `label-studio` - the main annotation tool
- `label-studio-ml` - the ML backend framework
- `ultralytics` - YOLOv8 and YOLOv5 models
- `segment-anything` - SAM segmentation model
- `opencv-python` - image processing
- `python-dotenv` - for environment variable management

## 3. Configure the ML backend

Copy `.env.example` → `.env` and fill in your Label Studio access token.

| Setting                     | Description                                                   |
|-----------------------------|---------------------------------------------------------------|
| `YOLO_MODEL`                | `yolo26n.pt.default` (fastest) … `yolo26l.pt` (most accurate) |
| `SAM_MODEL`                 | `sam2.1_t.pt` (tiny) … `sam2.1_l.pt` (large)                  |
| `CONF_THRESHOLD`            | Min detection confidence (0.0–1.0)                            |
| `LABEL_STUDIO_URL`          | Where Label Studio is running                                 |
| `LABEL_STUDIO_ACCESS_TOKEN` | From Label Studio → Account & Settings → Access Token         |

## 4. Start Label Studio

Open a **separate terminal** (keep it running) and run:

```powershell
`set DATA_UPLOAD_MAX_NUMBER_FILES=5000` <br/>

label-studio start
```

Label Studio will open at **http://localhost:8080**.

1. Create an account / log in
2. Create a new project
3. Upload some images (drag & drop or from local folder)

> **Tip:** You can also use the cell below to start Label Studio from the notebook.

## 5. Optional: Set the labeling config in LabelStudio

In your project → **Settings → Labeling Interface → Code**, paste the XML below.

> Customize the `<Label value="...">` list to match your actual class names.
> The backend auto-maps YOLO class names to labels — they must be identical.

```xml
<View>
  <Image name="image" value="$image"/>

  <!-- SAM segmentation output (primary) -->
  <PolygonLabels name="polygon_labels" toName="image" strokeWidth="2" opacity="0.5">
    <Label value="person"     background="#FF6B6B"/>
    <Label value="car"        background="#4ECDC4"/>
    <Label value="bicycle"    background="#45B7D1"/>
    <Label value="motorcycle" background="#96CEB4"/>
    <Label value="bus"        background="#FFEAA7"/>
    <Label value="truck"      background="#DDA0DD"/>
    <Label value="dog"        background="#98FB98"/>
    <Label value="cat"        background="#FFB347"/>
  </PolygonLabels>

  <!-- YOLO bbox fallback -->
  <RectangleLabels name="bbox_labels" toName="image" strokeWidth="2">
    <Label value="person"     background="#FF6B6B"/>
    <Label value="car"        background="#4ECDC4"/>
    <Label value="bicycle"    background="#45B7D1"/>
    <Label value="motorcycle" background="#96CEB4"/>
    <Label value="bus"        background="#FFEAA7"/>
    <Label value="truck"      background="#DDA0DD"/>
    <Label value="dog"        background="#98FB98"/>
    <Label value="cat"        background="#FFB347"/>
  </RectangleLabels>
</View>
```

## 6. Start the ML backend server

Open another **separate terminal** and run:

```powershell
cd C:\Users\Tymur\Projects\ComputerVision\Annotation\ml_backend
..\..\..venv\Scripts\python.exe server.py
```

You should see:
```
INFO  Loading YOLO model: yolo26n.pt
INFO  Loading SAM model: sam2.1_b.pt
INFO  YoloSamBackend ready.
 * Running on http://0.0.0.0:9090
```

Or run it from this notebook (background thread):

## 7. Connect the ML backend to Label Studio

In Label Studio:

1. Go to your project → **Settings → Machine Learning**
2. Click **Add Model**
3. Fill in:
   - **URL:** `http://localhost:9090`
   - **Name:** `YOLO26n + SAM2` (any name)
4. Click **Validate and Save** — you should see ✅ *Connected*
5. Enable **"Use for interactive pre-annotations"**

---

## Step 9 — Run auto-annotation

**Option A — Batch (all tasks at once):**
- Go to the task list → select all → **Actions → Retrieve Predictions**

**Option B — Single task (interactive):**
- Open any task → click the **Auto-Annotate** (⚡) button in the toolbar
- Label Studio sends the image to the ML backend and renders the predictions

You can then correct the polygons/boxes and submit the annotation.

---

## Troubleshooting

| Problem | Fix |
|---|---|
| *Connection refused* on port 9090 | Make sure `server.py` is running |
| *No predictions* returned | Check `LABEL_STUDIO_ACCESS_TOKEN` in `.env` |
| Class labels not matching | Edit `<Label value="...">` in config to match YOLO class names |
| Too many/few detections | Adjust `CONF_THRESHOLD` in `.env` (lower = more detections) |
| SAM not producing masks | Try switching to `sam2.1_t.pt` (faster) or check GPU memory |
| `ImportError: find_loader` | Run: `pip install django-environ==0.14.0` |

---

## Customisation

| Goal | How |
|---|---|
| Use a heavier model | Set `YOLO_MODEL=yolo26l.pt` in `.env` |
| Faster SAM | Set `SAM_MODEL=sam2.1_t.pt` or `mobile_sam.pt` |
| Custom YOLO weights | Set `YOLO_MODEL=/path/to/best.pt` |
| Different tag names in config | Edit `POLYGON_FROM_NAME`, `IMAGE_TO_NAME` in `.env` |

# TODO: Add re-training module ( after manual corrections, export annotations and train YOLO/SAM on them )